# Question 8 — Quelle politique par type de séjour, et qui remplit la semaine ?

**Auteur : Wenceslas Bouity** — Projet Valmont, Datamining B3 (ECE Paris), sujet 9.

La question 7 a montré que Valmont a **quatre types de séjours** et non les trois segments déclarés. Cette
question en tire les conséquences commerciales : pour chaque type, qu'est-ce qu'il rapporte, qu'est-ce qu'il
coûte en annulations, quand remplit-il l'hôtel, et quelle condition de vente faut-il lui appliquer ?

Elle répond aussi à une question précise de la direction : **qui remplit les nuits de semaine ?**

**Méthode** : profil chiffré de chaque type (prix, annulation, services, part en semaine), décomposition du
revenu perdu, puis une décision par type avec son gain estimé et sa plage de sensibilité.

**Point de départ** : `data/clean/valmont_clean.csv` et `data/clean/q7_types_sejours.csv`, le fichier de types
produit par la question 7.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import chi2_contingency

pd.set_option("display.width", 210)
pd.set_option("display.max_columns", 40)

# Memes couleurs qu'en Q7, attachees au type et non a son rang :
# un type garde sa couleur d'un graphique a l'autre et d'un notebook a l'autre.
COULEUR_TYPE = {
    "Court séjour de dernière minute": "#00b894",
    "Long séjour individuel": "#f2a600",
    "Groupe et séminaire": "#257de0",
    "Réservation très anticipée": "#ec4647",
}
# Ordre de lecture : du type le plus sur au plus risque (taux d'annulation croissant, Q7).
ORDRE = list(COULEUR_TYPE)
# Etiquettes courtes, pour les axes des graphiques seulement.
COURT = {"Court séjour de dernière minute": "Court séjour\ndernière minute",
         "Long séjour individuel": "Long séjour\nindividuel",
         "Groupe et séminaire": "Groupe et\nséminaire",
         "Réservation très anticipée": "Réservation\ntrès anticipée"}
ENCRE, ENCRE_2, GRILLE = "#191919", "#666666", "#dcdcdc"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRILLE, "axes.labelcolor": ENCRE, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlecolor": ENCRE, "axes.grid": True,
    "grid.color": GRILLE, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": ENCRE_2, "ytick.color": ENCRE_2, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

reservations = pd.read_csv("../data/clean/valmont_clean.csv")
types_q7 = pd.read_csv("../data/clean/q7_types_sejours.csv")

# Jointure interne : les 6 anomalies de la Q9 n'ont pas de type et sortent ici,
# exactement comme elles etaient sorties de la classification en Q7.
sejours = reservations.merge(types_q7, on="reservation_id", how="inner")

print("Réservations du fichier propre :", len(reservations))
print("Réservations typées en Q7      :", len(types_q7))
print("Après jointure                 :", len(sejours))
print("Perdues dans la jointure       :", len(types_q7) - len(sejours))
print("\nEffectif par type :")
print(sejours["type_sejour"].value_counts().reindex(ORDRE).to_string())

Réservations du fichier propre : 800
Réservations typées en Q7      : 794
Après jointure                 : 794
Perdues dans la jointure       : 0

Effectif par type :
type_sejour
Court séjour de dernière minute    363
Long séjour individuel             171
Groupe et séminaire                134
Réservation très anticipée         126


## 1. Contrôle de la jointure

La jointure est faite sur `reservation_id`, identifiant unique vérifié en question 7. Les 6 réservations
marquées comme anomalies n'ont pas de type : elles sont écartées ici comme elles l'étaient de la
classification, et restent traitées en question 9. Aucune ligne n'est perdue autrement, et les effectifs par
type sont identiques à ceux de la question 7 — c'est ce que contrôle la cellule ci-dessus.

In [2]:
# Deux grandeurs de revenu servent dans tout le notebook, definies une fois ici.
# revenu_potentiel : ce que la reservation aurait rapporte en chambre si elle avait ete honoree.
# Le prix est un prix par nuit et par chambre (verifie dans le notebook 00), d'ou le produit.
sejours["revenu_potentiel"] = sejours["prix_nuit_eur"] * sejours["nb_nuits"]
sejours["revenu_realise"] = sejours["revenu_potentiel"] * (1 - sejours["a_annule"])
sejours["revenu_perdu"] = sejours["revenu_potentiel"] * sejours["a_annule"]
sejours["petit_dejeuner_oui"] = (sejours["petit_dejeuner"] == "oui").astype(int)
sejours["en_semaine"] = 1 - sejours["weekend"]

print("Sur les", len(sejours), "réservations analysées :")
print(f"  revenu potentiel : {sejours['revenu_potentiel'].sum():>10,.0f} €")
print(f"  revenu réalisé   : {sejours['revenu_realise'].sum():>10,.0f} €")
print(f"  revenu annulé    : {sejours['revenu_perdu'].sum():>10,.0f} €  "
      f"({sejours['revenu_perdu'].sum() / sejours['revenu_potentiel'].sum():.1%} du potentiel)")
print(f"\n  nuitées vendues  : {sejours.loc[sejours['a_annule'] == 0, 'nb_nuits'].sum():>6,.0f}")
print(f"  nuitées annulées : {sejours.loc[sejours['a_annule'] == 1, 'nb_nuits'].sum():>6,.0f}")

Sur les 794 réservations analysées :
  revenu potentiel :    454,228 €
  revenu réalisé   :    338,948 €
  revenu annulé    :    115,279 €  (25.4% du potentiel)

  nuitées vendues  :  1,720
  nuitées annulées :    620


## 2. La fiche chiffrée des quatre types

C'est le tableau de référence de la question : tout ce qu'un directeur commercial doit savoir sur un type pour
décider de ses conditions de vente.

In [3]:
fiche = sejours.groupby("type_sejour").agg(
    reservations=("type_sejour", "size"),
    personnes=("nb_personnes", "mean"),
    nuits=("nb_nuits", "mean"),
    delai_median=("delai_reservation_j", "median"),
    prix_nuit=("prix_nuit_eur", "mean"),
    panier=("revenu_potentiel", "mean"),
    taux_annulation=("a_annule", "mean"),
    part_en_semaine=("en_semaine", "mean"),
    petit_dejeuner=("petit_dejeuner_oui", "mean"),
).reindex(ORDRE)

fiche["part_reservations"] = fiche["reservations"] / len(sejours)
fiche["revenu_realise"] = sejours.groupby("type_sejour")["revenu_realise"].sum().reindex(ORDRE)
fiche["revenu_perdu"] = sejours.groupby("type_sejour")["revenu_perdu"].sum().reindex(ORDRE)
fiche["part_revenu_realise"] = fiche["revenu_realise"] / fiche["revenu_realise"].sum()
fiche["part_revenu_perdu"] = fiche["revenu_perdu"] / fiche["revenu_perdu"].sum()
fiche["perdu_par_reservation"] = fiche["revenu_perdu"] / fiche["reservations"]

colonnes_profil = ["reservations", "part_reservations", "personnes", "nuits", "delai_median",
                   "prix_nuit", "panier", "taux_annulation", "part_en_semaine", "petit_dejeuner"]
print("Profil des quatre types :")
print(fiche[colonnes_profil].round(3).to_string())

Profil des quatre types :
                                 reservations  part_reservations  personnes  nuits  delai_median  prix_nuit   panier  taux_annulation  part_en_semaine  petit_dejeuner
type_sejour                                                                                                                                                           
Court séjour de dernière minute           363              0.457      1.350  2.116          13.0    198.715  414.676            0.187            0.680           0.466
Long séjour individuel                    171              0.215      1.789  4.912          25.0    199.244  965.371            0.228            0.678           0.386
Groupe et séminaire                       134              0.169     11.537  2.328          54.5    185.127  441.318            0.336            0.769           0.485
Réservation très anticipée                126              0.159      2.706  3.333          96.0    192.412  630.836            0.437      

In [4]:
def intervalle_wilson(succes, effectif, z=1.96):
    """Intervalle de confiance a 95 % d'une proportion, methode de Wilson.

    Preferee a la formule normale classique : elle reste valide sur de petits
    effectifs (ici 126 reservations pour le plus petit type) et ne sort jamais de [0, 1].
    """
    p = succes / effectif
    denominateur = 1 + z ** 2 / effectif
    centre = (p + z ** 2 / (2 * effectif)) / denominateur
    demi_largeur = z * np.sqrt(p * (1 - p) / effectif + z ** 2 / (4 * effectif ** 2)) / denominateur
    return centre - demi_largeur, centre + demi_largeur

print("Taux d'annulation et intervalle de confiance à 95 % :")
bornes = {}
for type_sejour in ORDRE:
    bloc = sejours[sejours["type_sejour"] == type_sejour]
    bas, haut = intervalle_wilson(bloc["a_annule"].sum(), len(bloc))
    bornes[type_sejour] = (bas, haut)
    print(f"  {type_sejour:33s} n = {len(bloc):3d}   {bloc['a_annule'].mean():.1%}   "
          f"IC95 [{bas:.1%} ; {haut:.1%}]")

print("\nLes intervalles du type le plus sûr et du type le plus risqué se recouvrent-ils ?",
      "non" if bornes[ORDRE[0]][1] < bornes[ORDRE[3]][0] else "oui")

Taux d'annulation et intervalle de confiance à 95 % :
  Court séjour de dernière minute   n = 363   18.7%   IC95 [15.1% ; 23.1%]
  Long séjour individuel            n = 171   22.8%   IC95 [17.2% ; 29.7%]
  Groupe et séminaire               n = 134   33.6%   IC95 [26.1% ; 41.9%]
  Réservation très anticipée        n = 126   43.7%   IC95 [35.3% ; 52.4%]

Les intervalles du type le plus sûr et du type le plus risqué se recouvrent-ils ? non


**Lecture.** Les quatre types ne sont pas quatre nuances d'un même comportement, ils demandent des
conditions différentes :

- **Court séjour de dernière minute** — 46 % des réservations, le type le plus sûr (18,7 % d'annulation), un
  panier de 415 €. C'est le socle du volume.
- **Long séjour individuel** — 22 % des réservations mais un panier de **965 €**, deux fois et demie celui du
  court séjour, pour un risque à peine plus élevé (22,8 %). C'est le type le plus rentable de Valmont.
- **Groupe et séminaire** — 17 % des réservations, 33,6 % d'annulation, et **77 % de ses nuits en semaine**,
  la proportion la plus forte des quatre.
- **Réservation très anticipée** — 16 % des réservations et 43,7 % d'annulation : plus de deux réservations
  sur cinq ne viennent jamais.

Les intervalles de confiance du type le plus sûr (15,1 % – 23,1 %) et du plus risqué (35,3 % – 52,4 %) ne se
recouvrent pas : l'écart n'est pas un effet d'échantillonnage. En revanche ceux du court et du long séjour
individuels se recouvrent largement — **ces deux types ne se distinguent pas par le risque, mais par le
panier**, et c'est exactement ce qui justifie qu'ils reçoivent des politiques différentes : l'un se pilote en
volume, l'autre en valeur.